# Dự Báo Mức Tiêu Thụ Tài Nguyên Của SQL Query Từ Estimated Execution Plan

Notebook trình bày toàn bộ pipeline: mỗi phần gọi script tương ứng trong `src/` bằng `%run`, sau đó hiển thị bảng và biểu đồ kết quả. Logic xử lý nằm trong `src/`; notebook chỉ điều phối và trình bày.

**Cách dùng:** mở notebook từ thư mục gốc của repo, chạy lần lượt từ trên xuống. Đọc `docs/LABEL_RULES.md` (quy tắc và biện luận label) trước khi chạy phần 4.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))   # để import được config và common
import common
import config

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Thu muc goc:", config.ROOT)

## 0. Kiểm tra môi trường


In [ ]:
%run src/00_check_env.py

## 1. Dữ Liệu

Mỗi lần chạy là một folder trong `data/raw/` gồm `plan_estimated.txt`, `plan_allstats.txt`, `plan_stats.csv`, `summary.json` và các file tham khảo. Xem một folder mẫu:

In [ ]:
# sorted(...) sắp xếp theo tên (tức theo thời gian chạy); [0] lấy folder đầu tiên.
first = sorted(config.RAW_DIR.iterdir())[0]
print("Folder mau:", first.name)
print("Cac file:", [f.name for f in first.iterdir()])
print()
print(first.joinpath("plan_estimated.txt").read_text(encoding="utf-8")[:2500])   # 2500 ký tự đầu

## 2. Feature Extraction

Đọc `plan_estimated.txt` (feature) và `summary.json` (số liệu để labeling) của từng lần chạy; đối chiếu cấu trúc estimated plan với actual plan.

In [ ]:
%run src/01_extract_features.py

In [ ]:
features = pd.read_csv(config.FEATURES_CSV)
check = pd.read_csv(config.CHECK_REPORT_CSV)
display(Markdown(f"**{len(features)} bản ghi, {len(common.FEATURE_COLS)} feature**"))
display(features.head())
display(check["structure"].value_counts().rename("so_lan_chay").to_frame())

## 3. Tiền Xử Lý Và EDA

Kiểm tra giá trị thiếu, loại feature hằng và feature hiếm, phát hiện outlier (chỉ báo cáo), phân phối elapsed time, CPU time và disk reads, Spearman correlation giữa estimated time và elapsed time thực tế, loại các cặp feature trùng lặp. Kết quả ghi vào `data/interim/features_clean.csv`.

In [ ]:
%run src/02_eda.py

In [ ]:
clean = pd.read_csv(config.FEATURES_CLEAN_CSV)
display(Markdown(f"**{len(clean)} bản ghi, {clean.shape[1]} cột sau tiền xử lý**"))
display(pd.read_csv(config.TAB_EDA / "outliers.csv", index_col=0).round(2))
display(pd.read_csv(config.TAB_EDA / "percentiles.csv", index_col=0).round(3))
display(pd.read_csv(config.TAB_EDA / "spearman_voi_elapsed.csv", index_col=0).round(2))
display(pd.read_csv(config.TAB_EDA / "feature_filter.csv").round(2))
for name in ["01_phan_phoi_elapsed_cpu.png", "02_time_uoc_luong_vs_thuc_te.png",
             "03_tuong_quan_dac_trung.png"]:
    display(Markdown(f"**{name}**"))
    display(Image(filename=str(config.FIG_EDA / name), width=650))

## 4. Gán Label Theo Quy Tắc

Gán label tự động theo quy tắc và ngưỡng đã biện luận trong `docs/LABEL_RULES.md`. Mỗi dòng của `data/labeled/labels.csv` có cột `label` và cột `rule` (quy tắc đã kích hoạt). Script cũng dời từng ngưỡng ±10% và đo tỉ lệ bản ghi đổi label.

In [ ]:
%run src/03_label_data.py

In [ ]:
labels = pd.read_csv(config.LABELS_CSV)
display(labels["label"].value_counts().rename("so_mau").to_frame())
display(labels["rule"].value_counts().rename("so_lan_kich_hoat").to_frame())
display(Markdown("**Độ nhạy của ngưỡng (dời ±10%)**"))
display(pd.read_csv(config.SENSITIVITY_CSV, index_col=0).round(3))

## 5. Training Và Evaluation

Stratified Group K-Fold (5 fold × 10 lần lặp, chia theo query), so sánh 4 mô hình (cây quyết định, hồi quy logistic, Random Forest, Gradient Boosting) với 2 baseline. Kèm các phép kiểm tra: nhãn bị xáo trộn, bỏ các lần chạy ngày cao điểm, chia fold không phân tầng, learning curve và permutation importance.

In [ ]:
%run src/04_train_evaluate.py

In [ ]:
for name, title in [("results_summary.csv", "Kết quả chính (tất cả lần chạy)"),
                    ("results_without_peak.csv", "Bỏ các lần chạy ngày cao điểm"),
                    ("results_unstratified.csv", "Chia fold không phân tầng"),
                    ("results_shuffled.csv", "Nhãn bị xáo trộn")]:
    display(Markdown(f"**{title}**"))
    display(pd.read_csv(config.TAB_RESULTS / name, index_col=0).round(3))
for img in sorted(config.FIG_RESULTS.glob("*.png")):
    display(Markdown(f"**{img.name}**"))
    display(Image(filename=str(img), width=650))

## 6. Streamlit App

App chạy trong terminal, không chạy trong notebook. Từ thư mục gốc của repo:

```
streamlit run app/streamlit_app.py
```

App mở tại `http://localhost:8501`: nạp một hoặc nhiều file `plan_estimated.txt` để dự báo và so sánh các phiên bản query.